In [165]:
from IPython.display import display, HTML

display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [166]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

In [167]:
import re
from sklearn.model_selection import train_test_split
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [168]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv(r"C:\Users\blown\Desktop\ironhack_AI_Engineering\8_NLP\labs\lab-natural-language-processing\data\kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [169]:
X = data.drop(columns='label')

y = data['label']

data_train, data_val, label_train, label_val = train_test_split(X, y, test_size=0.2, random_state=42)


## Data Preprocessing

In [170]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [171]:
def clean_text(text):

    text = text.lower()
    text = re.sub(r"^b['\"]", '', text)
    text = re.sub(r'<script\b[^<]*(?:(?!<\/script>)<[^<]*)*<\/script>', '', text, flags=re.IGNORECASE)
    text = re.sub(r'<style\b[^<]*(?:(?!<\/style>)<[^<]*)*<\/style>', '', text, flags=re.IGNORECASE)
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s[a-z]\s', ' ', text)
    text = re.sub(r'^[a-z]\b\s*', '', text)
    text = re.sub(r'\s+', ' ', text)

    return text

data_train['preprocessed_text'] = data_train['text'].apply(clean_text)
data_val['preprocessed_text'] = data_val['text'].apply(clean_text)

In [172]:
data_train['preprocessed_text'].head()

29      regards mr nelson smithkindly reply me on my ...
535    have not been able to reach oscar this am we a...
695     huma abedin bim checking with pat on the will...
557      can have it announced here on monday cant today
836     bank of africaagence san pedro bp san pedro c...
Name: preprocessed_text, dtype: str

## Now let's work on removing stopwords
Remove the stopwords.

In [173]:
stop_words_set = set(stopwords.words("english"))

def remove_stopwords(text):
    if not isinstance(text, str):
        return ""
    else:
        words = text.split()
        filtered_words = [word for word in words if word not in stop_words_set]
        return " ".join(filtered_words)

data_train['preprocessed_text'] = data_train['preprocessed_text'].apply(remove_stopwords)
data_val['preprocessed_text'] = data_val['preprocessed_text'].apply(remove_stopwords)

In [174]:
#check for missing values in dataset
print(data_train['preprocessed_text'].isna().sum())

0


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [175]:
lemmatizer = WordNetLemmatizer()

def lemmatize(text):
    words = text.split()
    words_with_tags = pos_tag(words)
    lemmatized_tokens = [lemmatizer.lemmatize(w, pos='a' if p[0] == "J" else 'v' if p[0] == "V" else 'r' if p[0] == "R" else 'n') for w, p in words_with_tags]
    return " ".join(lemmatized_tokens)

data_train['preprocessed_text'] = data_train['preprocessed_text'].apply(lemmatize)
data_val['preprocessed_text'] = data_val['preprocessed_text'].apply(lemmatize)

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [181]:
from collections import Counter
ham_messages = data_train['preprocessed_text'][label_train == 0]

spam_messages = data_train['preprocessed_text'][label_train == 1]


In [182]:
def most_common(words):
    wordcount = {}
    for word in words:
        if word not in wordcount:
            wordcount[word] = 1
        else:
            wordcount[word] += 1
    return sorted(wordcount.items(), key=lambda item:item[1], reverse=True)[:10]

ham_words = " ".join(ham_messages).split()
spam_words = " ".join(spam_messages).split()
print(f'Ham:', most_common(ham_words))
print(f'Spam:', most_common(spam_words))

Ham: [('u', 101), ('say', 96), ('would', 93), ('call', 92), ('state', 92), ('pm', 89), ('work', 87), ('president', 84), ('percent', 76), ('get', 72)]
Spam: [('money', 795), ('account', 670), ('bank', 615), ('fund', 568), ('u', 444), ('transfer', 435), ('business', 393), ('contact', 369), ('transaction', 347), ('country', 334)]


## Extra features

In [183]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,preprocessed_text,money_mark,suspicious_words,text_len
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",regard mr nelson smithkindly reply private ema...,0,0,75
535,I have not been able to reach oscar this am. W...,able reach oscar suppose send pdb receive,0,0,41
695,; Huma Abedin B6I'm checking with Pat on the 5...,huma abedin bim check pat work jack jake resta...,0,0,76
557,I can have it announced here on Monday - can't...,announce monday cant today,0,0,26
836,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,bank africaagence san pedro bp san pedro cote ...,1,1,1039


## How would work the Bag of Words with Count Vectorizer concept?

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vectorizer = CountVectorizer()
X_Bow = count_vectorizer.fit_transform(data_train['preprocessed_text'])



['aac' 'aaclocated' 'aae' ... 'zumadirector' 'zumae' 'zurich']


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [189]:
tfidf_vectorizer = TfidfVectorizer()
tfidf = tfidf_vectorizer.fit_transform(data_train['preprocessed_text'])

## And the Train a Classifier?

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code